# 01 - Combined Extract-Load-Build (with Pipeline)

End-to-end: read CSV → **transform pipeline** (RowToNode + InferEdges) → write to Neptune.

In [ ]:
import os, csv
from graphrag_toolkit.lexical_graph.storage import GraphStoreFactory
from document_graph import Node
from document_graph.graph_build import node_to_cypher, edge_to_cypher
from document_graph.query import DocumentGraphQueryEngine
from document_graph.transform.transformer_provider_config import TransformerProviderConfig
from document_graph.transform.graph_transformers.row_to_node import RowToNodeTransformer
from document_graph.transform.graph_transformers.infer_edges import EdgeInferencer

## Step 1: Read CSV

In [ ]:
csv_path = os.path.expanduser('~/SageMaker/document-graph/data/users.csv')
with open(csv_path) as f:
    records = list(csv.DictReader(f))
print(f'Read {len(records)} records')
print(records[0])

## Step 2: Transform — RowToNode

In [ ]:
config = TransformerProviderConfig(name='row_to_node', args={'type': 'User'})
row_to_node = RowToNodeTransformer(config)
nodes = row_to_node.transform(records)
print(f'Transformed to {len(nodes)} nodes')
print(f'First node: {nodes[0]}')

## Step 3: Transform — InferEdges

In [ ]:
config = TransformerProviderConfig(name='infer_edges', args={'source_field': 'account_id', 'edge_type': 'SAME_ACCOUNT'})
edge_inferencer = EdgeInferencer(config)
edges = edge_inferencer.transform(records)
print(f'Inferred {len(edges)} edges')
if edges:
    print(f'First edge: {edges[0]}')

## Step 4: Build Cypher + Write to Neptune

In [ ]:
GRAPH_STORE = 'neptune-db://obs-app-dev-graph.cluster-czupj1ab2tc6.us-east-1.neptune.amazonaws.com:8182'
gs = GraphStoreFactory.for_graph_store(GRAPH_STORE).__enter__()
TENANT = 'demo'

# Write nodes
for n in nodes:
    node = Node(id=n.get('id', n.get('_id', '')), labels=[n.get('node_type', 'Row')], properties=n)
    cypher, params = node_to_cypher(node, tenant_id=TENANT)
    gs.execute_query(cypher, params)
print(f'✅ Wrote {len(nodes)} nodes to Neptune (tenant={TENANT})')

## Step 5: Query back

In [ ]:
engine = DocumentGraphQueryEngine(gs, tenant_id=TENANT)
result = engine.get_nodes('User')
print(f'Found {len(result)} User nodes:')
for r in result:
    print(f'  {r}')